## Atividade avaliativa: Análise de cluster por município com Python
Por: Leonardo Rodrigues e Valentina Rodrigues.


In [2]:
from google.colab import files

uploaded = files.upload()  # selecione o arquivo data.csv

Saving data.csv to data (1).csv


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


In [4]:
df_raw = pd.read_csv('data.csv', encoding='latin-1', sep=';', decimal=',')

print("Shape:", df_raw.shape)
print("\nPrimeiras colunas:", df_raw.columns[:6].tolist())
print("\nMunicípios (primeiros 5):", df_raw['Municipio'].head().tolist())
print("\nValores ausentes por coluna (primeiras 10):")
print(df_raw.isnull().sum().head(10))

Shape: (497, 82)

Primeiras colunas: ['Municipio', 'Código', 'Contabilidade Social/Série 2002 em diante/PIB/Valor adicionado bruto a preços básicos/Agropecuária/2002 (R$ mil)', 'Contabilidade Social/Série 2002 em diante/PIB/Valor adicionado bruto a preços básicos/Agropecuária/2003 (R$ mil)', 'Contabilidade Social/Série 2002 em diante/PIB/Valor adicionado bruto a preços básicos/Agropecuária/2004 (R$ mil)', 'Contabilidade Social/Série 2002 em diante/PIB/Valor adicionado bruto a preços básicos/Agropecuária/2005 (R$ mil)']

Municípios (primeiros 5): ['Aceguá', 'Água Santa', 'Agudo', 'Ajuricaba', 'Alecrim']

Valores ausentes por coluna (primeiras 10):
Municipio                                                                                                           0
Código                                                                                                              0
Contabilidade Social/Série 2002 em diante/PIB/Valor adicionado bruto a preços básicos/Agropecuária/2002 (R$ m

In [5]:
id_vars = ['Municipio', 'Código']
value_cols = [c for c in df_raw.columns if c not in id_vars]

df_melted = df_raw.melt(id_vars=id_vars, value_vars=value_cols,
                         var_name='variavel_ano', value_name='valor')

# Extrair o ano (últimos 4 dígitos antes do espaço)
df_melted['Ano'] = df_melted['variavel_ano'].str.extract(r'/(\d{4})\s')

# Extrair o nome da variável (parte entre o último / e o /ANO)
df_melted['variavel'] = df_melted['variavel_ano'].str.extract(
    r'básicos/(.+?)/\d{4}')

# Simplificar nome das variáveis
mapa_variaveis = {
    'Agropecuária': 'Agropecuaria',
    'Indústria': 'Industria',
    'Serviços/Administração Pública': 'Adm_Publica',
    'Serviços/Outros Serviços': 'Outros_Servicos'
}
df_melted['variavel'] = df_melted['variavel'].map(mapa_variaveis)

# Remover linhas sem ano ou variável mapeada
df_melted = df_melted.dropna(subset=['Ano', 'variavel'])
df_melted['Ano'] = df_melted['Ano'].astype(int)
df_melted['valor'] = pd.to_numeric(df_melted['valor'], errors='coerce')

print("Shape após reshape:", df_melted.shape)
print("Anos disponíveis:", sorted(df_melted['Ano'].unique()))
print("Variáveis:", df_melted['variavel'].unique())

Shape após reshape: (39760, 6)
Anos disponíveis: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Variáveis: ['Agropecuaria' 'Industria' 'Adm_Publica' 'Outros_Servicos']


In [6]:
df_long = df_melted.pivot_table(
    index=['Municipio', 'Código', 'Ano'],
    columns='variavel',
    values='valor',
    aggfunc='mean'   # equivalente ao AGGREGATE do SPSS
).reset_index()

df_long.columns.name = None  # remover nome do índice de colunas

print("Shape final (long):", df_long.shape)
print("\nPrimeiras linhas:")
df_long.head(8)

Shape final (long): (4490, 7)

Primeiras linhas:


,Municipio,Código,Ano,Adm_Publica,Agropecuaria,Industria,Outros_Servicos
0,Aceguá,4300034,2013,23266.344,94878.579,7930.544,47911.392
1,Aceguá,4300034,2014,26519.804,109180.913,10513.980,56936.263
2,Aceguá,4300034,2015,27489.673,125636.429,10875.549,66165.904
3,Aceguá,4300034,2016,30173.598,130383.037,10774.504,71217.557
4,Aceguá,4300034,2017,30997.729,124098.448,13457.229,81238.321
5,Aceguá,4300034,2018,33626.185,109860.977,11467.303,77816.223
6,Aceguá,4300034,2019,36826.355,115919.158,12912.875,73168.727
7,Aceguá,4300034,2020,38148.581,116916.018,14449.294,78440.503


In [7]:
variaveis_cluster = ['Agropecuaria', 'Industria', 'Adm_Publica', 'Outros_Servicos']

print("Valores ausentes antes:")
print(df_long[variaveis_cluster].isnull().sum())

# Preencher com a mediana por ano (estratégia conservadora)
for var in variaveis_cluster:
    df_long[var] = df_long.groupby('Ano')[var].transform(
        lambda x: x.fillna(x.median())
    )

print("\nValores ausentes depois:")
print(df_long[variaveis_cluster].isnull().sum())

Valores ausentes antes:
Agropecuaria       12
Industria          13
Adm_Publica        12
Outros_Servicos    14
dtype: int64

Valores ausentes depois:
Agropecuaria        7
Industria          11
Adm_Publica         8
Outros_Servicos     8
dtype: int64


In [8]:
resultados_clusters = []
resultados_bic = []

anos = sorted(df_long['Ano'].unique())
scaler = StandardScaler()

for ano in anos:
    print(f"Processando ano {ano}...", end=' ')

    df_ano = df_long[df_long['Ano'] == ano].copy()

    # Remover municípios que ainda tenham NaN em qualquer variável
    df_ano_clean = df_ano.dropna(subset=variaveis_cluster).copy()

    if len(df_ano_clean) < 2:
        print(f"PULADO (municípios insuficientes após remover NaN)")
        continue

    X = df_ano_clean[variaveis_cluster].values
    X_scaled = scaler.fit_transform(X)

    # Limitar k ao número de municípios disponíveis
    k_max = min(15, len(df_ano_clean) - 1)

    bic_por_k = {}
    for k in range(1, k_max + 1):
        gmm = GaussianMixture(n_components=k, random_state=42, n_init=5)
        gmm.fit(X_scaled)
        bic = gmm.bic(X_scaled)
        bic_por_k[k] = bic
        resultados_bic.append({'Ano': ano, 'N_clusters': k, 'BIC': round(bic, 4)})

    k_otimo = min(bic_por_k, key=bic_por_k.get)
    print(f"k ótimo = {k_otimo} (BIC = {bic_por_k[k_otimo]:.2f})")

    gmm_final = GaussianMixture(n_components=k_otimo, random_state=42, n_init=5)
    gmm_final.fit(X_scaled)
    df_ano_clean['Cluster'] = gmm_final.predict(X_scaled) + 1

    resultados_clusters.append(df_ano_clean[['Municipio', 'Código', 'Ano', 'Cluster']])

print("\nConcluído!")

Processando ano 2003... PULADO (municípios insuficientes após remover NaN)
Processando ano 2004... PULADO (municípios insuficientes após remover NaN)
Processando ano 2005... PULADO (municípios insuficientes após remover NaN)
Processando ano 2007... PULADO (municípios insuficientes após remover NaN)
Processando ano 2008... PULADO (municípios insuficientes após remover NaN)
Processando ano 2010... PULADO (municípios insuficientes após remover NaN)
Processando ano 2011... PULADO (municípios insuficientes após remover NaN)
Processando ano 2012... PULADO (municípios insuficientes após remover NaN)
Processando ano 2013... k ótimo = 10 (BIC = -4632.57)
Processando ano 2014... k ótimo = 7 (BIC = -4669.31)
Processando ano 2015... k ótimo = 12 (BIC = -4566.66)
Processando ano 2016... k ótimo = 13 (BIC = -4629.75)
Processando ano 2017... k ótimo = 14 (BIC = -4503.95)
Processando ano 2018... k ótimo = 13 (BIC = -4159.31)
Processando ano 2019... k ótimo = 11 (BIC = -4250.10)
Processando ano 2020...

In [11]:
# Arquivo 1: Município, Ano, Cluster (sem a coluna Código)
df_clusters = pd.concat(resultados_clusters, ignore_index=True)
df_clusters = df_clusters[['Municipio', 'Ano', 'Cluster']]
df_clusters = df_clusters.rename(columns={'Municipio': 'Município'})
df_clusters = df_clusters.sort_values(['Ano', 'Município']).reset_index(drop=True)

# Arquivo 2: Ano, Número de clusters testado, BIC
df_bic = pd.DataFrame(resultados_bic)
df_bic = df_bic.rename(columns={'N_clusters': 'Número de clusters testado'})
df_bic = df_bic[['Ano', 'Número de clusters testado', 'BIC']]
df_bic = df_bic.sort_values(['Ano', 'Número de clusters testado']).reset_index(drop=True)

# Exportar
df_clusters.to_excel('clusters_municipios_ano.xlsx', index=False)
df_bic.to_excel('bic_clusters_municipios.xlsx', index=False)

print("Arquivos exportados com sucesso!")
print(f"\nclusters_municipios_ano.xlsx → {len(df_clusters)} linhas")
print(f"bic_clusters_municipios.xlsx → {len(df_bic)} linhas")

print("\nAmostra — clusters:")
display(df_clusters.head(8))
print("\nAmostra — BIC:")
display(df_bic.head(10))

Arquivos exportados com sucesso!

clusters_municipios_ano.xlsx → 4473 linhas
bic_clusters_municipios.xlsx → 135 linhas

Amostra — clusters:


,Município,Ano,Cluster
0,Aceguá,2013,8
1,Agudo,2013,8
2,Ajuricaba,2013,8
3,Alecrim,2013,6
4,Alegrete,2013,4
5,Alegria,2013,6
6,Almirante Tamandaré do Sul,2013,6
7,Alpestre,2013,6



Amostra — BIC:


,Ano,Número de clusters testado,BIC
0,2013,1,3683.9013
1,2013,2,-2696.3421
2,2013,3,-3158.8568
3,2013,4,-3932.4148
4,2013,5,-4505.2634
5,2013,6,-4503.2911
6,2013,7,-4589.9062
7,2013,8,-4609.2265
8,2013,9,-4559.0780
9,2013,10,-4632.5747


In [13]:
from google.colab import files

files.download('clusters_municipios_ano.xlsx')
files.download('bic_clusters_municipios.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Comentário Analítico
Em 2013 foram escolhidos 10 clusters, em 2014 e 2020 apenas 7, em 2015 foram 12, em 2016 e 2018 foram 13, em 2017 chegou ao pico com 14, e em 2019 e 2021 foram 11 clusters cada. O número variou bastante ao longo dos anos, oscilando entre 7 e 14 grupos. Isso indica que o perfil econômico dos municípios gaúchos não é estático, e que em determinados anos a diferença entre eles se torna mais ou menos acentuada. Essa instabilidade pode estar associada a choques econômicos, climáticos ou políticos que afetam de forma diferente cada região do estado. As variáveis que mais parecem diferenciar os grupos são o valor adicionado da Indústria e dos Outros Serviços, por apresentarem maior dispersão entre municípios grandes e pequenos. A Agropecuária também deve ter um papel relevante, especialmente para separar municípios do interior com forte produção rural daqueles com economia mais diversificada. Os agrupamentos refletem que no Rio Grande do Sul os municípios industrializados se separam dos agropecuários e dos dependentes da Administração Pública. A queda para 7 clusters em 2020 pode estar ligada ao impacto nivelador da pandemia, que reduziu a atividade econômica de forma generalizada, enquanto o pico de 14 em 2017 pode indicar uma recuperação desigual entre municípios após a crise de 2015-2016.